In [12]:
# https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=wJpXpmjEYC_T
# https://www.bilibili.com/video/BV1BbFaeVE4W  PyTorch手搓Transformer
# https://github.com/hankinghu/literature-books/tree/master

In [1]:

import torch
import torch.nn as nn
from torch.nn import functional as F
import textwrap
import random

# 超参数
file_name="sanguo-all.txt"
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
wrap_width = 50
max_iters = 10000
eval_interval = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.1
# ------------

torch.manual_seed(1337)


In [2]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open(file_name, 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [3]:
# Head类 注意力机制
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__ ()
        self.query = nn.Linear(n_embd,head_size,bias=False) # 线性变换层
        self.key = nn.Linear(n_embd,head_size,bias=False) # 线性变换层
        self.value = nn.Linear(n_embd,head_size,bias=False) # 线性变换层
        self.register_buffer("tril",torch.tril(torch.ones(block_size,block_size)))#不可训练的,结构(约等于常量)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        q= self.query(x)  #(B,T, head size)
        k=self.key(x)
        wei =q @ k.transpose(-2,-1)*k.shape[-1]**-0.5 #注意力方阵(B，T，T)
        wei = wei.masked_fill(self.tril == 0,float("-inf"))# 掩码填充
        wei =F.softmax(wei,dim=-1)
        wei = self.dropout(wei)  # 随机去掉(归零)一些值，增加网络的稳定性
        v = self.value(x)
        out = wei @ v   #(B, T, head size)
        return out

In [4]:
# 语言模型
class LanguageModel(nn.Module):
    def __init__ (self):
        super().__init__ ()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.head = Head(n_embd)
        self.network = nn.Sequential(
            nn.Linear(n_embd,256),
            nn.ReLU(),
            nn.Linear(256,vocab_size)
        )
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

    def forward(self,idx,targets=None):
        B,T=idx.shape     #(B,T)B= batch_size,T= block_size，数据为token(整数)形式
        token_embd=self.token_embedding_table(idx)
        position_idx= torch.arange(T,device=device)
        position_embd = self.position_embedding_table(position_idx)
        x=token_embd + position_embd #(B,T,n embd)
        head_out = self.head(x)  #添加注意力头
        logits =self.network(head_out)  #(B，T，vocab size)
        if targets is None:
            loss = None
        else:
            B, T, C= logits.shape
            logits =logits.view(B*T, C)  #摊平
            targets = targets.view(B*T)
            loss=F.cross_entropy(logits, targets)

        # B,T=idx.shape #B= batch size,T= block size，数据为token(整数)形式
        # random_tensor = torch.rand(B,T,vocab_size,device=device) #
        # logits = random_tensor /random_tensor.sum(dim=-1, keepdim=True)
        # loss = None
        return logits, loss
    
    def generate(self, token_sequ, max_new_tokens):
        # token_sequ已知的上文,max_new_tokens是续写的长度(B，T)
        for _ in range(max_new_tokens):
            tokens_input = token_sequ[:, -block_size: ]
            logits, loss = self.forward(tokens_input)  # logits,(B, T, vocab size)
            logits = logits[:,-1,:] #只取字符串最后一个,(概率分布向量格式)
            probs =F.softmax(logits,dim=-1)
            token_next = torch.multinomial(probs,num_samples=1)# 概率分布向量-->one-hot 向量-->整数token
            token_sequ =torch.cat((token_sequ, token_next), dim=1)
        new_tokens =token_sequ[:,-max_new_tokens:]
        return new_tokens

In [5]:
#--损失评测--------
@torch.no_grad()   #不做梯度计算的decorator,作用域为整个函数
def estimate_loss(model):
    out = {}
    model.eval()  #把模型转化为evaluate模式(默认模式是train)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)  # 建立一个初始值为0的容器,用于储存loss值
        for k in range(eval_iters):
            X, Y = get_batch(split)  # split是一个字符串,用来控制get_batch()函数的行为
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()  # out是含有两个元素的字典，一个是train，一个是val，每个元素对应一个loss的平均值
    model.train() # 再转化为训练模式(如果之前没有转为evaluate模式,则不需要这一步,因为模型建立后默认为训练模式)
    return out

In [6]:
def main():
    print(f"训练内容:{file_name}")
    model =LanguageModel()#实例化
    model = model.to(device)
    print(sum(p.numel()for p in model.parameters())/1e6,'M parameters')# 打印有多少个参数
    #设定一个优化器
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
    # 训练循环
    for i in range(max_iters):
        if i % eval_interval ==0 or i==max_iters - 1:
            losses =estimate_loss(model)
            print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        #取样
        xb,yb = get_batch("train")
        logits, loss=model(xb, yb)   #前馈运算
        optimizer.zero_grad()   #把旧的梯度归零
        loss.backward()   #反向传播,计算新的梯度
        optimizer.step()  #做一步优化计算

    print("训练结束，下面开始生成内容")
    max_new_tokens =200
    start_idx = random.randint(0, len(val_data)-block_size-max_new_tokens)
    #上文内容
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)# (B, T)B = 1,T = block size
    context[0,:]=val_data[start_idx:start_idx+block_size]
    context_str =decode(context[0].tolist())#一阶张量
    wrapped_context_str = textwrap.fill(context_str, width=wrap_width)
    #真实下文
    real_next_tokens = torch.zeros((1,max_new_tokens), dtype=torch.long, device=device)
    real_next_tokens[0, :]= val_data[start_idx+block_size: start_idx+block_size+max_new_tokens]
    real_next_tokens_str = decode(real_next_tokens[0].tolist())# 一阶张量
    wrapped_real_next_tokens_str = textwrap.fill(real_next_tokens_str, width=wrap_width)
    #生成下文
    generated_tokens = model.generate(context, max_new_tokens)
    generated_str =decode(generated_tokens[0].tolist())
    wrapped_generated_str = textwrap.fill(generated_str, width=wrap_width)

    print("---------上文内容---------:")
    print(wrapped_context_str)
    print("---------真实上文内容---------:")
    print(wrapped_real_next_tokens_str)
    print("---------生成内容---------:")
    print(wrapped_generated_str)


main()

训练内容:sanguo-all.txt
1.30342 M parameters
step 0: train loss 8.2809, val loss 8.2807
step 1000: train loss 4.9449, val loss 5.4958
step 2000: train loss 4.5458, val loss 5.3225
step 3000: train loss 4.3294, val loss 5.2594
step 4000: train loss 4.1975, val loss 5.2542
step 5000: train loss 4.0862, val loss 5.2820
step 6000: train loss 4.0010, val loss 5.3120
step 7000: train loss 3.9331, val loss 5.3689
step 8000: train loss 3.8840, val loss 5.4165
step 9000: train loss 3.8270, val loss 5.4573
step 9999: train loss 3.7945, val loss 5.5272
训练结束，下面开始生成内容
---------上文内容---------:
唤子邓忠分付曰：“汝用心守把此处，任他搦战，却勿轻出。吾今夜引兵
---------真实上文内容---------:
去祁山救应。” 是夜二更，姜维正在寨中设计，忽听得寨外喊声震地，鼓角喧天，人报邓艾引三千精兵夜战。诸
将欲出，维止之曰：“勿得妄动。”原来邓艾引兵至蜀寨前哨探了一遍，乘势去救祁山，邓忠自入城去了。姜维唤
诸将曰：“邓艾虚作夜战之势，必然去救祁山寨矣。”乃唤傅佥分付曰：“汝守此寨，勿轻与敌。”嘱毕，维自引
三千兵来助张翼。 却说张翼正到祁山攻打，守寨将师纂兵少，支持不住。看看待破，忽然邓艾兵至，冲杀了一阵
---------生成内容---------:
十余，主公子韩遂视之，失守御其死，可敌黄忠：吾粮草，我却若得一个优客；而充山也。”，遂手执刀，立斩了
营；两下从武阳军，势穷力往荆州与绍为救宫。”孔明见东连忙救得王，陶谦横刀，押眉掀在身，雷声石。瑜就计
我，令张引一军，不可轻敌，借荆州来看见诸洞数，眉告曰：“辅